# Phase 2: teach the translator everyday Kikamba (back-translation)

1. Collect the Kikamba sentences read in Digital Divide Data's public Kamba speech dataset (CC-BY-4.0): about
   161,000 recordings of everyday topics, de-duplicated. The 300 everyday test sentences are left out.
2. **Back-translate** them: the current model (`lawmaluki/nllb-200-1.3B-kamba-ft`) translates each into English.
3. Fine-tune the current model again on *machine English → real Kikamba* plus all the earlier data, so it learns
   to **write** natural everyday Kikamba.
4. Compare the new model with the current one on FLORES devtest and on the held-out children's stories.

About **3–4 hours** on a T4. **Before running (Kaggle):** right panel → *Session options*: **GPU T4 x2** (or P100),
**Internet On**, *Persistence: Files only*; menu **Add-ons → Secrets → `HF_TOKEN`** (a write token), attached to this
notebook. Then **Run All** and keep the tab open, or **Save Version → Save & Run All (Commit)**. Every step saves its
result, so a rerun after an interruption continues where it stopped.

In [ ]:
# 0. Choices
INIT_MODEL = "lawmaluki/nllb-200-1.3B-kamba-ft"           # the model the app uses now
CT2_MODEL = "lawmaluki/nllb-200-1.3B-kamba-ft-ct2-int8"   # the same model, fast, for back-translating
BT_MAX = 40000      # back-translated sentence pairs to train on
EPOCHS = 1

In [ ]:
# 1. Setup (~3 min). The token is passed as an environment variable (no huggingface_hub import before pip).
import os, socket, torch
assert torch.cuda.is_available(), "No GPU: Session options -> Accelerator -> GPU T4 x2"
try:
    socket.create_connection(("huggingface.co", 443), timeout=10).close()
except OSError:
    raise SystemExit("No internet: Session options -> Internet -> On")
token = os.environ.get("HF_TOKEN")
try:
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret("HF_TOKEN")
except Exception:
    pass
if not token:
    raise SystemExit("Add your Hugging Face token: Add-ons -> Secrets -> HF_TOKEN (and attach it to this notebook)")
os.environ["HF_TOKEN"] = token

WORK = "/kaggle/working" if os.path.isdir("/kaggle/working") else "/content"
os.chdir(WORK)
if not os.path.isdir("understand-kamba"):
    !git clone -q --depth 1 -b phase2-backtranslation https://github.com/lawmaluki/understand-kamba.git || git clone -q --depth 1 https://github.com/lawmaluki/understand-kamba.git
os.chdir(f"{WORK}/understand-kamba")
!pip uninstall -y -q torchao
!pip install -q "transformers>=4.45" peft sacrebleu sentencepiece accelerate ctranslate2 pyarrow
import huggingface_hub
print("Ready:", torch.cuda.get_device_name(0), "| huggingface_hub", huggingface_hub.__version__)

In [ ]:
# 2. Collect DDD's Kikamba sentences (~20-40 min: streams the 146 GB dataset, keeps only the text)
import csv, json, time, unicodedata
from concurrent.futures import ThreadPoolExecutor
import pyarrow as pa
from huggingface_hub import HfApi, HfFileSystem

SENT = f"{WORK}/ddd_sentences.tsv"
DDD = "gamamima/Kamba-ASR-Data-Subset-484H"
if not os.path.exists(SENT):
    fs = HfFileSystem()
    files = sorted(f.rfilename for f in HfApi().dataset_info(DDD).siblings if f.rfilename.endswith(".arrow"))
    def read(path):
        out = []
        with fs.open(f"datasets/{DDD}/{path}", "rb", block_size=32 * 2**20) as h:
            for b in pa.ipc.open_stream(h):
                t, th = b.column("transcripts").to_pylist(), b.column("theme").to_pylist()
                out += [(x, y) for x, y in zip(t, th) if x]
        return out
    seen, start, done = {}, time.time(), 0
    with ThreadPoolExecutor(8) as pool:
        for rows in pool.map(read, files):
            for text, theme in rows:
                t = " ".join(unicodedata.normalize("NFC", text).split())
                seen.setdefault(t, theme)
            done += 1
            if done % 25 == 0 or done == len(files):
                print(f"{done}/{len(files)} files, {len(seen):,} unique sentences, {(time.time() - start) / 60:.0f} min", flush=True)
    with open(SENT, "w", encoding="utf-8", newline="") as f:
        w = csv.writer(f, delimiter="\t"); w.writerow(["kam", "theme"]); w.writerows(seen.items())
with open(SENT, encoding="utf-8") as f:
    sentences = list(csv.DictReader(f, delimiter="\t"))
print(f"{len(sentences):,} unique Kikamba sentences")
try:  # keep a private copy for later rounds (needs a write token)
    HfApi().create_repo("lawmaluki/kamba-ddd-sentences", repo_type="dataset", private=True, exist_ok=True)
    HfApi().upload_file(path_or_fileobj=SENT, path_in_repo="ddd_sentences.tsv", repo_id="lawmaluki/kamba-ddd-sentences", repo_type="dataset")
    print("saved privately to lawmaluki/kamba-ddd-sentences")
except Exception as e:
    print("(could not save a copy to Hugging Face:", type(e).__name__, ")")

In [ ]:
# 3. Back-translate them into English with the current model (~15-30 min on the GPU)
import random, ctranslate2
from huggingface_hub import snapshot_download
from transformers import AutoTokenizer

BT = f"{WORK}/ddd_bt.tsv"
if not os.path.exists(BT):
    with open("eval/everyday/sentences.tsv", encoding="utf-8") as f:
        test = {r["kam"].strip() for r in csv.DictReader(f, delimiter="\t")}
    pool = [s["kam"] for s in sentences if s["kam"] not in test and 3 <= len(s["kam"].split()) <= 40]
    random.Random(1234).shuffle(pool)
    pool = pool[: int(BT_MAX * 1.1)]
    path = snapshot_download(CT2_MODEL)
    tok = AutoTokenizer.from_pretrained(path)
    tok.src_lang = "kam_Latn"
    try:  # fast path: CTranslate2 on the GPU (needs NVIDIA libraries Kaggle may lack)
        model = ctranslate2.Translator(path, device="cuda", compute_type="int8_float16")
        def translate(chunk):
            toks = [tok.convert_ids_to_tokens(tok(s)["input_ids"]) for s in chunk]
            res = model.translate_batch(toks, target_prefix=[["eng_Latn"]] * len(toks), beam_size=4,
                                        no_repeat_ngram_size=4, max_batch_size=64, max_decoding_length=256)
            return [tok.decode(tok.convert_tokens_to_ids(r.hypotheses[0][1:]), skip_special_tokens=True).strip() for r in res]
        translate(["Wĩ mũseo?"])
        print("back-translating with CTranslate2 on the GPU")
    except Exception as e:  # reliable path: the same model in transformers, float16 on the GPU
        print("CTranslate2 GPU unavailable (", type(e).__name__, ") - using transformers instead (slower)")
        from transformers import AutoModelForSeq2SeqLM
        model = AutoModelForSeq2SeqLM.from_pretrained(INIT_MODEL, torch_dtype=torch.float16).to("cuda").eval()
        tok = AutoTokenizer.from_pretrained(INIT_MODEL); tok.src_lang = "kam_Latn"
        def translate(chunk):
            out_ = []
            for j in range(0, len(chunk), 32):
                enc = tok(chunk[j:j + 32], return_tensors="pt", padding=True, truncation=True, max_length=128).to("cuda")
                with torch.inference_mode():
                    gen = model.generate(**enc, forced_bos_token_id=tok.convert_tokens_to_ids("eng_Latn"), num_beams=4,
                                         no_repeat_ngram_size=4, max_new_tokens=enc["input_ids"].shape[1] * 2 + 16)
                out_ += [d.strip() for d in tok.batch_decode(gen, skip_special_tokens=True)]
            return out_
    out, start = [], time.time()
    for i in range(0, len(pool), 256):
        out += translate(pool[i:i + 256])
        if (i // 256) % 20 == 0:
            print(f"{len(out):,}/{len(pool):,} translated, {(time.time() - start) / 60:.0f} min", flush=True)
    with open(BT, "w", encoding="utf-8", newline="") as f:
        w = csv.writer(f, delimiter="\t"); w.writerow(["kam", "eng"])
        w.writerows((k, e) for k, e in zip(pool, out) if e)
    del model
with open(BT, encoding="utf-8") as f:
    bt = list(csv.DictReader(f, delimiter="\t"))
print(f"{len(bt):,} back-translated pairs, e.g.:")
for r in bt[:5]:
    print("  ", r["kam"], "\n   ->", r["eng"])

In [ ]:
# 4. Fine-tune and compare (~2-2.5 h: scores the current model, filters web pairs, trains, scores the new model).
#    A progress line every ~2 minutes; the full log is in train_log.txt.
import subprocess, re
log = open(f"{WORK}/train_log.txt", "a")
cmd = ["python", "training/finetune_nllb.py", "--model", INIT_MODEL, "--out", f"{WORK}/nllb-kam-ft-v2",
       "--bt-file", BT, "--bt-max", str(BT_MAX), "--epochs", str(EPOCHS)]
proc = subprocess.Popen(cmd, stdout=log, stderr=subprocess.STDOUT, env=dict(os.environ, CUDA_VISIBLE_DEVICES="0", PYTHONUNBUFFERED="1"))
while proc.poll() is None:
    time.sleep(120)
    lines = [l for l in open(f"{WORK}/train_log.txt", encoding="utf-8", errors="replace").read().splitlines() if re.match(r"\d\d:\d\d:\d\d ", l)]
    if lines:
        print(lines[-1], flush=True)
log.close()
if proc.returncode != 0 or not os.path.exists(f"{WORK}/nllb-kam-ft-v2/eval_report.json"):
    lines = open(f"{WORK}/train_log.txt", encoding="utf-8", errors="replace").read().splitlines()
    print(chr(10).join(l for l in lines[-40:] if "it/s]" not in l))
    raise SystemExit(f"Stopped (exit code {proc.returncode}) - copy the lines above to Claude")

In [ ]:
# 5. Results: current model vs new model
r = json.load(open(f"{WORK}/nllb-kam-ft-v2/eval_report.json"))
print(f"{'test set / direction':28} {'current':>8} {'new':>8}")
better = 0
for k, v in r["finetuned"].items():
    b = r.get("base", {}).get(k)
    print(f"{k:28} {b if b is not None else '-':>8} {v:>8}  {'' if b is None else f'{v - b:+.2f}'}")
    better += b is not None and v > b
d = json.load(open(f"{WORK}/nllb-kam-ft-v2/test_outputs.json", encoding="utf-8"))
print()
for (kam, eng), hyp in list(zip(d["tests"]["stories"], d["outputs"]["stories/en_to_kam"]))[:8]:
    print("ENGLISH  :", eng); print("REFERENCE:", kam); print("NEW MODEL:", hyp); print()
print("Upload only if the new model is better on the important rows (especially stories/en_to_kam).")

## 6. (Optional) Upload the new model

Set `UPLOAD = True` only if step 5 shows it is better. It is saved **privately** as
`lawmaluki/nllb-200-1.3B-kamba-ft-v2`; tell Claude, who converts it to the fast CPU format and switches the app.

In [ ]:
UPLOAD = False
if UPLOAD:
    from huggingface_hub import HfApi
    repo = "lawmaluki/nllb-200-1.3B-kamba-ft-v2"
    HfApi().create_repo(repo, private=True, exist_ok=True)
    HfApi().upload_folder(folder_path=f"{WORK}/nllb-kam-ft-v2", repo_id=repo,
                          allow_patterns=["*.json", "*.safetensors", "*.model", "*.txt"],
                          ignore_patterns=["checkpoints/*", "test_outputs.json", "opus_kept_*.json"])
    print("uploaded: https://huggingface.co/" + repo)